# Sistema di Visione 3D

# 02 Calibrazione Camera 

In [449]:
import os
import cv2 as cv
import numpy as np
import pickle
import random
from matplotlib import pyplot as plt
from os import listdir, path

# Directory e costanti
IMGDIR      = 'images'
TARGET      = os.path.join(IMGDIR,'target')
CENTERS     = os.path.join(TARGET,'centers')
CROP        = os.path.join(TARGET,'cropped')
OUTDIR      = 'outputs'  # Directory per salvare i risultati
MAX_ROWS    = 14         # Numero massimo di righe della griglia
MAX_COLS    = 21         # Numero massimo di colonne della griglia
TARGET_STEP = 20         # Distanza tra i cerchi in mm.
Z_BASE      = 50         # Altezza della base in mm

if not path.exists(CENTERS):
    os.makedirs(CENTERS)

In [450]:
def vardump(var, fname):
    """
    Serializza un oggetto Python e lo salva in un file binario.

    Parametri:
      var   : Oggetto Python da salvare su disco.
      fname : Percorso e nome del file binario in cui salvare 'var'.

    Note:
      - Utilizza la libreria 'pickle' per la serializzazione.
      - La modalità 'wb' indica scrittura in binario.
    """
    with open(fname, 'wb') as fw:
        pickle.dump(var, fw)

## Find Circles (with Hough) & Main Targets

In [451]:
def find_first_valid_circles_grid(img_gray, max_rows = MAX_ROWS, max_cols = MAX_COLS):
    """
    Cerca il primo pattern di cerchi valido in un'immagine in scala di grigi,
    partendo dalle dimensioni di griglia più grandi e decrescendo fino a
    trovare un match.

    Parametri:
      img_gray (np.array): immagine in scala di grigi.
      max_rows (int)     : numero massimo di righe (default=20).
      max_cols (int)     : numero massimo di colonne (default=30).

    Ritorna:
      best_pattern (tuple): pattern (righe, colonne) del primo pattern trovato.
      centers      (np.array): coordinate dei centri trovati, shape (n, 1, 2).
      found_num    (int)      : numero di cerchi trovati per il pattern.

    Note:
      - Utilizza cv.findCirclesGrid con la modalità CALIB_CB_SYMMETRIC_GRID.
      - Restituisce None se non individua alcun pattern.
    """
    # Inizia dai pattern più grandi verso i più piccoli
    for c in range(max_cols, 8, -1):
        for r in range(max_rows, 8, -1):
            pattern = (c, r)
            ret, centers = cv.findCirclesGrid(
                img_gray,
                pattern,
                flags=cv.CALIB_CB_SYMMETRIC_GRID
            )
            if ret and centers is not None:
                return pattern, centers, len(centers)

    # Se non viene trovato alcun pattern
    return None, None, 0

In [452]:
def findMainTarget(imgGray, centers, m):
    """
    Trova i principali target (cerchi) in un'immagine in scala di grigi utilizzando la trasformata di Hough per cerchi.

    Parametri:
      imgGray (np.array): immagine in scala di grigi.
      centers (np.array): array di centri dei cerchi trovati.
      m       (int)     : parametro per il calcolo della dimensione della ROI.

    Ritorna:
      main_targets (list): lista dei principali target trovati.
    """
    radii = []
    for center in centers:
        x, y = int(center[0][0]), int(center[0][1])
        
        # Definizione della ROI con margine per evitare il taglio di cerchi sfocati
        roi_size = max(50, int(imgGray.shape[0] / (2 * m)))
        padding = 15
        x_start = max(0, x - roi_size - padding)
        x_end   = min(imgGray.shape[1], x + roi_size + padding)
        y_start = max(0, y - roi_size - padding)
        y_end   = min(imgGray.shape[0], y + roi_size + padding)
        
        roi = imgGray[y_start:y_end, x_start:x_end]

        # Parametri Hough più stringenti per ridurre i falsi positivi
        circles = cv.HoughCircles(
            roi,
            cv.HOUGH_GRADIENT,
            dp=1.1,
            minDist=roi_size / 2,
            param1=80,    # Soglia Canny più alta
            param2=20,    # Soglia accumulatore maggiore
            minRadius=8,  # Raggio minimo dei cerchi
            maxRadius=roi_size  # Raggio massimo dei cerchi
        )
        
        if circles is not None:
            circles = np.uint16(np.around(circles[0]))
            # Prende il raggio più grande per evitare piccoli falsi positivi
            radius = max(c[2] for c in circles)
            radii.append(radius)
        else:
            radii.append(0)
    
    # Associa i centri ai rispettivi raggi e ordina per raggio decrescente
    centers_with_radii = [(c, r) for c, r in zip(centers, radii) if r > 0]
    centers_with_radii.sort(key=lambda x: x[1], reverse=True)
    main_targets = [np.uint16(c[0]) for c, r in centers_with_radii[:3]]
    
    # Visualizzazione dei principali target trovati
    debug_img = cv.cvtColor(imgGray, cv.COLOR_GRAY2BGR)
    for pt in main_targets:
        cv.circle(debug_img, (pt[0], pt[1]), 8, (255, 0, 255), -1)
    
    plt.imshow(debug_img[..., ::-1])
    plt.title("Principali Target Trovati")
    plt.axis("off")
    plt.show()

    return main_targets

## Find Axis Dircetion

In [453]:
def findAxisDirection(img, centers, mainTargets):
    """
    Trova la direzione dell'asse principale basata sui bersagli principali trovati.

    Parametri:
      img (np.array)         : immagine in scala di grigi.
      centers (np.array)     : array di centri dei cerchi trovati.
      mainTargets (list)     : lista dei principali bersagli trovati.

    Ritorna:
      mainCircle (np.array)  : coordinate del centro principale calcolato.
      iCenter (int)          : indice del centro più vicino al centro principale.
    """
    h, w = img.shape
    mainTargets = np.array(mainTargets)  # Converte la lista dei bersagli principali in un array numpy.

    # Calcolo dei bordi per il raggruppamento lungo l'asse X (colonne)
    mean_i = np.mean(mainTargets[:, 0])  # Media delle coordinate X dei bersagli principali.
    edges_i = np.array([0, mean_i, w])  # Bordi per suddividere l'asse X in due intervalli: [0, mean_i] e [mean_i, w].
    idxs_i = np.digitize(mainTargets[:, 0], edges_i)  # Assegna ciascun punto al bin corrispondente.

    # Selezione del bin lungo l'asse X
    bin_i = 1 if sum(idxs_i == 1) == 2 else 2  # Se ci sono esattamente due punti nel primo bin, seleziona il primo bin, altrimenti il secondo.
    ypts = mainTargets[idxs_i == bin_i, :]  # Filtra i punti assegnati al bin selezionato lungo l'asse X.

    # Calcolo dei bordi per il raggruppamento lungo l'asse Y (righe)
    mean_j = np.mean(mainTargets[:, 1])  # Media delle coordinate Y dei bersagli principali.
    edges_j = np.array([0, mean_j, h])  # Bordi per suddividere l'asse Y in due intervalli: [0, mean_j] e [mean_j, h].
    idxs_j = np.digitize(mainTargets[:, 1], edges_j)  # Assegna ciascun punto al bin corrispondente.

    # Selezione del bin lungo l'asse Y
    bin_j = 1 if sum(idxs_j == 1) == 2 else 2  # Se ci sono esattamente due punti nel primo bin, seleziona il primo bin, altrimenti il secondo.
    xpts = mainTargets[idxs_j == bin_j, :]  # Filtra i punti assegnati al bin selezionato lungo l'asse Y.

    # Calcolo del centro medio basato sui punti selezionati
    mainCircle = np.array([np.mean(ypts[:, 0]), np.mean(xpts[:, 1])])  # Combina la media delle coordinate X e Y per determinare il centro principale.

    # Individua il centro più vicino al "mainCircle"
    iCenter = np.linalg.norm(centers - np.ones(centers.shape) * mainCircle, axis=2).argmin()  # Calcola la distanza euclidea tra ogni centro rilevato e il centro principale.

    return mainCircle, iCenter

## Camera Calibration

In [ ]:
# Liste per calibrazione
objpoints = []  # Punti reali 3D
imgpoints = []  # Punti immagine 2D

# Scansione di tutte le immagini nella directory TARGET
for f in sorted(os.listdir(CROP)):
    if f.endswith('.jpg'):
        # Lettura dell'immagine
        img_path = os.path.join(CROP, f)
        img_bgr = cv.imread(img_path, cv.IMREAD_COLOR)
        img_gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)
        assert img_bgr is not None, f"Errore: Immagine {f} non trovata!"

        # Altezza della telecamera (zsg) estratta dal nome file
        zsg = int(f[:2]) * 10  # mm

        # Trova il pattern corretto
        pattern, centers, found_num = find_first_valid_circles_grid(img_gray)
        if pattern is None:
            print(f"Nessun pattern trovato in {f}")
            continue
        else:
            # Calcolo del numero totale di cerchi previsti
            total_cerchi = pattern[0] * pattern[1]
            print(f"Primo pattern valido trovato in {f}: {pattern[1]} righe x {pattern[0]} colonne")
            print(f"Cerchi effettivamente trovati: {found_num} / {total_cerchi}")

        # Trova i principali target
        mainTargets = findMainTarget(img_gray, centers, pattern[0])
        print(f"Principali target trovati: {mainTargets}")

        # Trova la direzione dell'asse
        mainCircle, iCenter = findAxisDirection(img_gray, centers, mainTargets)
        print(f"Centro principale calcolato: {mainCircle}")
        print(f"Centro più vicino: {centers[iCenter]}")

        reference_center = centers[iCenter] # salvataggio variabile per corretta stampa
        
        # Generazione dei punti 3D reali (objp)
        objp = np.zeros((pattern[0] * pattern[1], 3), np.float32)
        objp[:, :2] = np.mgrid[0:pattern[0], 0:pattern[1]].T.reshape(-1, 2)  # Genera le coordinate 2D (x, y) del pattern.
        objp = objp - objp[iCenter]  # Centra i punti rispetto al centro del pattern.
        objp = objp * TARGET_STEP  # Scala i punti secondo la distanza reale tra i punti del pattern.
        objp[:, 2] = zsg + Z_BASE  # 50 mm di altezza della base del supporto

        # Disegno dei centri trovati
        img_result = img_bgr.copy()
        for iCenter in range(centers.shape[0]):
            center = centers[iCenter]
            cv.circle(img_result, tuple(center[0].astype(int)), 6, (255, 0, 255), -1)
            cv.putText(img_result, f"{int(objp[iCenter,0] / TARGET_STEP)}, {int(objp[iCenter,1] / TARGET_STEP)}", tuple(center[0].astype(int)), cv.FONT_HERSHEY_COMPLEX, 0.7, (0, 0, 255), 2)

        cv.circle(img_result, np.uint16(mainCircle), 10, (0, 255, 0), -1)
        cv.circle(img_result, np.uint16(reference_center)[0], 10, (255, 255, 0), -1)

        # Salvataggio dell'immagine con i centri
        out_path = os.path.join(CENTERS, f"centers_{f}")
        cv.imwrite(out_path, img_result)
        print(f"Immagine con centri salvata: {out_path}\n\n")

        # Aggiungi i punti per calibrazione
        objpoints.append(objp)
        imgpoints.append(centers)

## Camera Calibration

In [ ]:
# Stima iniziale della matrice intrinseca
h, w = img_gray.shape
focal_length = max(w, h)
center = (w / 2, h / 2)
initial_camera_matrix = np.array([
    [focal_length, 0, center[0]],
    [0, focal_length, center[1]],
    [0, 0, 1]
], dtype=np.float32)

# Calibrazione della telecamera
ret, cameraMatrix, distCoeffs, rvecs, tvecs = cv.calibrateCamera(
    objpoints, 
    imgpoints, 
    (w, h), 
    initial_camera_matrix, 
    None,
    flags=cv.CALIB_USE_INTRINSIC_GUESS
)

# Calcolo della nuova matrice della telecamera e della ROI
newCameraMatrix, roi = cv.getOptimalNewCameraMatrix(cameraMatrix, distCoeffs, (w, h), 1, (w, h))

# Visualizza i risultati della calibrazione
print("Matrice della telecamera:")
display(cameraMatrix)
print("Coefficienti di distorsione:")
display(distCoeffs)
print("Nuova matrice della telecamera:")
display(newCameraMatrix)

# Salvataggio dei risultati della calibrazione
vardump(cameraMatrix, os.path.join(OUTDIR, 'cameraMatrix.dump'))
vardump(distCoeffs, os.path.join(OUTDIR, 'distCoeffs.dump'))
vardump(newCameraMatrix, os.path.join(OUTDIR, 'newCameraMatrix.dump'))
vardump(roi, os.path.join(OUTDIR, 'roi.dump'))

# Estrazione dei parametri dalla nuova matrice della telecamera
cx = newCameraMatrix[0, 2]  # Coordinate del punto principale (x)
cy = newCameraMatrix[1, 2]  # Coordinate del punto principale (y)
fx = newCameraMatrix[0, 0]  # Lunghezza focale in pixel
print(f"Parametri della nuova matrice della telecamera: cx = {cx:.2f}, cy = {cy:.2f}, fx = {fx:.2f}")

# Calcolo della matrice inversa della telecamera
inverse = np.linalg.inv(newCameraMatrix)
print("Matrice inversa della nuova matrice della telecamera:")
display(inverse)

# Unione dei punti oggettivi e dei punti immagine
worldPoints = np.vstack(objpoints)
imagePoints = np.vstack(imgpoints)

# Risoluzione del problema di PnP (Pose Estimation)
print("Risoluzione del problema di PnP (Pose Estimation):")
ret, rvec1, tvec1 = cv.solvePnP(worldPoints, imagePoints, newCameraMatrix, distCoeffs)

# Visualizzazione dei vettori di rotazione e traslazione
print("Vettore di rotazione (rvec1):")
display(rvec1)
print("Vettore di traslazione (tvec1):")
display(tvec1)

# Conversione del vettore di rotazione in matrice di rotazione
print("Matrice di rotazione (R) ottenuta da rvec1:")
R_mtx, jac = cv.Rodrigues(rvec1)
display(R_mtx)

# Creazione della matrice estrinseca
print("Matrice estrinseca (R|t):")
Rt = np.column_stack((R_mtx, tvec1))
display(Rt)

# Calcolo della matrice di proiezione
print("Matrice di proiezione (newCamMtx * R|t):")
P_mtx = newCameraMatrix.dot(Rt)
display(P_mtx)

# Solve Problem

$$Q=\left[\begin{matrix}
x_{sg} & y_{sg} & z_{sg} & 1 & 0 & 0 & 0 & 0 & -x_{sg}\cdot i_s & -y_{sg}\cdot i_s & -z_{sg}\cdot i_s \\
0 & 0 & 0 & 0 & x_{sg} & y_{sg} & z_{sg} & 1 &   -x_{sg}\cdot j_s & -y_{sg}\cdot j_s & -z_{sg}\cdot j_s
\end{matrix}
\right]$$

$$\bar{m}=Q^+ \cdot \bar{d} \rightarrow Q^+=\left(Q^T\cdot Q\right)^{-1}\cdot Q^T$$

In [ ]:
# Matrice dei coefficienti del sistema lineare e vettore dei termini noti
Q = []
d = []

# Itera su tutti i punti del pattern
for i in range(worldPoints.shape[0]):
    wp = worldPoints[i]  # Coordinate 3D di un punto del mondo reale
    op = imagePoints[i][0]  # Coordinate 2D proiettate sul piano immagine
    print(f"Coordinate 3D del mondo reale: {wp}, Coordinate 2D proiettate: {op}")

    # Aggiunge un'equazione per la componente x
    Q.append(np.concatenate([wp, [1, 0, 0, 0, 0], -1 * wp * op[0]]))

    # Aggiunge un'equazione per la componente y
    Q.append(np.concatenate([[0, 0, 0, 0], wp, [1], -1 * wp * op[1]]))

    # Aggiunge le componenti x e y al vettore dei termini noti
    d.append(op[0])
    d.append(op[1])

# Converte le liste Q e d in array numpy
Q = np.array(Q)
d = np.array(d)

# Verifica le dimensioni del sistema lineare
print(f"Dimensioni del sistema lineare: Q = {Q.shape}, d = {d.shape}")

# Risolve il sistema lineare Q * m = d usando la pseudo-inversa di Q
m = np.dot(np.linalg.pinv(Q), d)
print("Coefficienti della matrice m:")
display(m)

# Calcola gamma^2, che è il modulo del vettore (m8, m9, m10)
gamma2 = m[8]**2 + m[9]**2 + m[10]**2 # coefficiente di proporzionalità

# Calcola la componente di traslazione z (tz) normalizzando il sistema
tz = np.sqrt(1 / gamma2)

# Costruisce la matrice di proiezione P (3x4) a partire dal vettore `m` normalizzato
P = np.reshape(np.concatenate([m, [1]]) * tz, (3, 4))
print("Matrice di proiezione P:")
display(P)

In [457]:
vardump(m, path.join(OUTDIR, 'm.dump'))  
# Salva il vettore `m`, contenente i coefficienti della matrice di proiezione, in un file binario.
# `OUTDIR` specifica la directory di output e il file si chiama `m.dump`.

vardump(P, path.join(OUTDIR, 'PMatrix.dump'))  
# Salva la matrice di proiezione `P` (3x4) in un file binario chiamato `PMatrix.dump`.

## Verifica di Calibrazione

In [ ]:
# Converti le liste in array numpy
objpoints = np.array(objpoints, dtype=object)
imgpoints = np.array(imgpoints, dtype=object)

# Numero totale di punti
num_points = len(objpoints)

# Percentuale di punti da destinare alla validazione
num_validation = max(1, int(num_points * 0.1))

# Indici casuali per la validazione
validation_indices = random.sample(range(num_points), num_validation)

# Indici per la calibrazione
calibration_indices = [i for i in range(num_points) if i not in validation_indices]

# Sottoinsiemi di calibrazione
objpoints_calib = objpoints[calibration_indices]
imgpoints_calib = imgpoints[calibration_indices]

# Sottoinsiemi di validazione
objpoints_valid = objpoints[validation_indices]
imgpoints_valid = imgpoints[validation_indices]

# Inizializza la matrice intrinseca e i coefficienti di distorsione
img_shape = img_gray.shape[::-1]

# Calibrazione della fotocamera
ret, cameraMatrix, distCoeffs, rvecs, tvecs = cv.calibrateCamera(
    objpoints_calib, 
    imgpoints_calib, 
    img_shape,
    initial_camera_matrix, 
    None,
    flags=cv.CALIB_USE_INTRINSIC_GUESS
)

print("\nCalibrazione completata con successo!")
print("Matrice intrinseca della telecamera:")
print(cameraMatrix)
print("Coefficienti di distorsione:")
print(distCoeffs)

# Ri-proiezione dei punti di validazione
total_error = 0
for i in range(len(objpoints_valid)):
    imgpoints2, _ = cv.projectPoints(objpoints_valid[i], rvecs[i], tvecs[i], cameraMatrix, distCoeffs)
    error = cv.norm(imgpoints_valid[i], imgpoints2, cv.NORM_L2) / len(imgpoints2)
    total_error += error

mean_error = total_error / len(objpoints_valid)
print(f"Errore medio di ri-proiezione (pixel): {mean_error}")

# Visualizzazione grafica su un set di validazione
plt.figure()
plt.scatter(imgpoints_valid[0].squeeze()[:,0], imgpoints_valid[0].squeeze()[:,1], c='r', marker='o', label='Misurati')
plt.scatter(imgpoints2.squeeze()[:,0], imgpoints2.squeeze()[:,1], c='g', marker='x', label='Proiettati')
plt.gca().invert_yaxis()
plt.title("Ri-proiezione Punti")
plt.legend()
plt.show()

In [ ]:
# ri-proiezione senza cv.projectPoints

# Conversione liste in array numpy
objpoints = np.array(objpoints, dtype=object)
imgpoints = np.array(imgpoints, dtype=object)

num_points = len(objpoints)

# Percentuale di punti da destinare alla validazione
num_validation = max(1, int(num_points * 0.1))

# Indici casuali per la validazione
validation_indices = random.sample(range(num_points), num_validation)

# Indici per la calibrazione
calibration_indices = [i for i in range(num_points) if i not in validation_indices]

# Sottoinsiemi di calibrazione
objpoints_calib = objpoints[calibration_indices]
imgpoints_calib = imgpoints[calibration_indices]

# Sottoinsiemi di validazione
objpoints_valid = objpoints[validation_indices]
imgpoints_valid = imgpoints[validation_indices]

# Inizializza la matrice intrinseca e i coefficienti di distorsione
img_shape = img_gray.shape[::-1]

# Calibrazione della fotocamera
ret, cameraMatrix, distCoeffs, rvecs, tvecs = cv.calibrateCamera(
    objpoints_calib, 
    imgpoints_calib, 
    img_shape,
    initial_camera_matrix, 
    None,
    flags=cv.CALIB_USE_INTRINSIC_GUESS
)

# Calcolo della nuova matrice della telecamera e della ROI
newCameraMatrix, roi = cv.getOptimalNewCameraMatrix(cameraMatrix, distCoeffs, (w, h), 1, (w, h))

# Estrazione dei parametri dalla nuova matrice della telecamera
cx = newCameraMatrix[0, 2]  # Coordinate del punto principale (x)
cy = newCameraMatrix[1, 2]  # Coordinate del punto principale (y)
fx = newCameraMatrix[0, 0]  # Lunghezza focale in pixel

# Calcolo della matrice inversa della telecamera
inverse = np.linalg.inv(newCameraMatrix)

# Unione dei punti oggettivi e dei punti immagine
worldPoints = np.vstack(objpoints_calib)
imagePoints = np.vstack(imgpoints_calib)

# Risoluzione del problema di PnP (Pose Estimation)
ret, rvec1, tvec1 = cv.solvePnP(worldPoints, imagePoints, newCameraMatrix, distCoeffs)

# Conversione del vettore di rotazione in matrice di rotazione
R_mtx, jac = cv.Rodrigues(rvec1)

# Creazione della matrice estrinseca
Rt = np.column_stack((R_mtx, tvec1))

# Calcolo della matrice di proiezione
P_mtx = newCameraMatrix.dot(Rt)

# Matrice dei coefficienti del sistema lineare e vettore dei termini noti
Q = []
d = []

# Itera su tutti i punti del pattern
for i in range(worldPoints.shape[0]):
    wp = worldPoints[i]  # Coordinate 3D di un punto del mondo reale
    op = imagePoints[i][0]  # Coordinate 2D proiettate sul piano immagine

    # Aggiunge un'equazione per la componente x
    Q.append(np.concatenate([wp, [1, 0, 0, 0, 0], -1 * wp * op[0]]))

    # Aggiunge un'equazione per la componente y
    Q.append(np.concatenate([[0, 0, 0, 0], wp, [1], -1 * wp * op[1]]))

    # Aggiunge le componenti x e y al vettore dei termini noti
    d.append(op[0])
    d.append(op[1])

# Converte le liste Q e d in array numpy
Q = np.array(Q)
d = np.array(d)

# Risolve il sistema lineare Q * m = d usando la pseudo-inversa di Q
m = np.dot(np.linalg.pinv(Q), d)

# Calcola gamma^2, che è il modulo del vettore (m8, m9, m10)
gamma2 = m[8]**2 + m[9]**2 + m[10]**2 # coefficiente di proporzionalità

# Calcola la componente di traslazione z (tz) normalizzando il sistema
tz = np.sqrt(1 / gamma2)

# Costruisce la matrice di proiezione P (3x4) a partire dal vettore `m` normalizzato
P = np.reshape(np.concatenate([m, [1]]) * tz, (3, 4))

# is, js imgpoints
# x,y,z sg objpoints

# Riproietta i punti di validazione
proj = []
# Ks = int(objpoints_valid[0][0][2])
objpoints_valid = np.vstack(objpoints_valid)
imgpoints_valid = np.vstack(imgpoints_valid)
total_error = 0
for i in range(len(objpoints_valid)):
    objp = objpoints_valid[i]
    imgp = imgpoints_valid[i]

    # Aggiungi una colonna di 1 ai punti 3D per la moltiplicazione con P
    objp_homogeneous = np.hstack((objp, 1))

    # Riproietta i punti 3D nel piano immagine
    imgp_proj_homogeneous = np.dot(P, objp_homogeneous.T).T

    # Converti le coordinate omogenee in coordinate 2D
    imgp_proj = imgp_proj_homogeneous[:2] / imgp_proj_homogeneous[2]
    proj.append(imgp_proj)
    # Calcola l'errore di riproiezione
    error = np.linalg.norm(imgp - imgp_proj)
    total_error += error

mean_error = total_error / len(objpoints_valid)
print(f"\nErrore medio di riproiezione: {mean_error}")

# Visualizzazione grafica su un set di validazione
proj = np.array(proj)
plt.figure()
plt.scatter(imgpoints_valid.squeeze()[:, 0], imgpoints_valid.squeeze()[:, 1], c='r', marker='o', label='Misurati')
plt.scatter(proj[:, 0], proj[:, 1], c='g', marker='x', label='Proiettati')
plt.gca().invert_yaxis()
plt.title("Ri-proiezione Punti")
plt.legend()
plt.show()

## Stima dei parametri

$$\left[\begin{matrix}I_S \\ J_s \\ K_s\end{matrix}\right]=
\left[
\begin{matrix}
\frac{(s_ufr_{11}+i_0r_{31})}{t_z} & \frac{(s_ufr_{12}+i_0r_{32})}{t_z} & \frac{(s_ufr_{13}+i_0r_{33})}{t_z} & \frac{(s_uft_x+i_0t_z)}{t_z} \\
\frac{(s_vfr_{21}+j 0r{31})}{t_z} & \frac{(s_vfr_{22}+j_0r_{32})}{t_z} & \frac{(s_vfr_{23}+j_0r_{33})}{t_z} & \frac{(s_vft_y+j_0t_z)}{t_z} \\
\frac{r_{31}}{t_z} & \frac{r_{32}}{t_z} & \frac{r_{33}}{t_z} & 1 
\end{matrix}  
\right]
\cdot\left[\begin{matrix}X_{sg} \\ Y_{sg} \\ Z_{sg} \end{matrix}\right]$$

In [ ]:
z_main = 0 # Altezza del target principale di default
for f in sorted(os.listdir(TARGET)):
        if f.lower().endswith('.jpg') and 'main' in f:
            z_main = int(f[:2]) * 10 + Z_BASE  # mm
            break

# Stima dei parametri i0 e j0
i0 = P[0][0] * P[2][0] + P[0][1] * P[2][1] + P[0][2] * P[2][2]
j0 = P[1][0] * P[2][0] + P[1][1] * P[2][1] + P[1][2] * P[2][2]
print(f"i0: {i0}, j0: {j0}")

# Stima di fx e fy
fx = np.sqrt(P[0][0]**2 + P[0][1]**2 + P[0][2]**2 - i0**2)
fy = np.sqrt(P[1][0]**2 + P[1][1]**2 + P[1][2]**2 - j0**2)
print(f"fx: {fx}, fy: {fy}")

# Stima della matrice di rotazione
r1 = (P[0, :3] - i0 * P[2, :3]) / fx
r2 = (P[1, :3] - j0 * P[2, :3]) / fy
r3 = P[2, :3]
R = np.vstack((r1, r2, r3))

# Normalizzazione della matrice di rotazione R
U, _, Vt = np.linalg.svd(R)
R = np.dot(U, Vt)

print("Matrice di rotazione stimata:")
print(R)

# Stima del vettore di traslazione
tz = z_main  # z main target
t = np.array([(P[0, 3] - i0 * tz) / fx, (P[1, 3] - j0 * tz) / fy, tz]).reshape(-1, 1)
print("Vettore di traslazione stimato:")
print(t)